# Interim results from a FelixGenome run in progress

Step-2 shards are written as they finish, so a run that is still going — or one
that failed before `Concat` and `Summarize` — already holds usable results. This
notebook reports per-phenotype chromosome coverage, merges whichever phenotypes
are complete, and plots the three arms, so Figure 3's layout can be settled
before the run ends.

Everything is streamed with `gsutil cat`; only the merged TSVs land on the VM.
Merging and shard bookkeeping are done by `scripts/felix_interim_scan.sh` — the
nested-scatter index and the highest-attempt rule live there and are tested, so
this notebook drives it rather than reimplementing it.

**Read the arms differently.** The ordinary arm is usually complete across all 22
chromosomes, so its λGC and Manhattan are the genome-wide numbers. The repeat
arms lag, because `RuVcf`/`BaVcf` gate them — their λGC is computed over a
**chromosome subset** and must be labelled that way, never quoted as
genome-wide.

**What the ordinary arm is and is not a comparator for.** It tests one marker
per VCF record; the repeat arms test one marker per catalog locus. The two
marker sets do not join, so "did the repeat branch beat the ordinary branch"
is not a p-value comparison between them. The comparison that answers the
question lives *inside* the repeat loci and is the 2×2 that
`felix/scripts/compare_ru_arms.py` builds from the two repeat arms:

| | pooled (`p.value_ancALL`) | ancestry-aware (`P_cct_admixed_c`) |
| --- | --- | --- |
| **carrier coding** | A — conventional GWAS baseline | B |
| **repeat dosage** | C | D — production arm |

Repeat-dosage benefit is D vs B and C vs A; local-ancestry benefit is D vs C
and B vs A. Each contrast is paired at the locus, with the same null model,
GRM, samples and carrier haplotypes, so a chromosome subset costs precision
but not validity. Significance *counts* are a different matter: they are not
interpretable on a subset, and per `felix/REPEAT_DOSAGE.md` "more hits" is not
a benefit claim even on a complete run without matched λ.

| Variable | Default | What it does |
| --- | --- | --- |
| `FELIX_RUN_ROOT` | — | `gs://…/FelixGenome/<workflow-id>`; or set `FELIX_SUBMISSION_ID` + `FELIX_WORKFLOW_ID` |
| `FELIX_PHENOTYPES` | `$WORKSPACE_BUCKET/tractor_mix_pilot/selected_phenotypes.txt` | scatter order; the inner shard index is a line number in this file |
| `FELIX_PHENO_COV` | `$WORKSPACE_BUCKET/tractor_mix_pilot/pheno_cov.tsv` | for case/control counts in the calibration table |
| `FELIX_MIN_CHROMS_MAIN` | `22` | merge a phenotype only at this coverage |
| `FELIX_MIN_CHROMS_REPEAT` | `1` | repeat arms are partial by nature; coverage is reported per phenotype |
| `FELIX_INTERIM_DIR` | `felix_interim` | working directory |

Point it at the run that produced the shards. For a rescued run either works —
the cached shards are copies — but the original is guaranteed to hold them all.


In [ ]:
from pathlib import Path
import os
import subprocess
import sys

for _d in (Path.cwd() / "scripts", Path.cwd().parent / "scripts", Path.cwd().parent.parent / "scripts"):
    if (_d / "terra_notebook.py").is_file():
        sys.path.insert(0, str(_d.resolve()))
        break
else:
    _bucket = os.environ.get("WORKSPACE_BUCKET", "").rstrip("/")
    if not _bucket:
        raise FileNotFoundError("scripts/ not found locally and WORKSPACE_BUCKET is unset.")
    _dest = (Path.cwd() / "scripts").resolve()
    _dest.mkdir(parents=True, exist_ok=True)
    subprocess.check_call(["gsutil", "-m", "rsync", "-r", f"{_bucket}/scripts/", str(_dest) + "/"])
    sys.path.insert(0, str(_dest))

from terra_notebook import init_notebook
SCRIPTS = init_notebook("felix_interim_scan.sh")


def sh(cmd: str, check: bool = True) -> int:
    """Run a shell command and fail on a non-zero exit.

    get_ipython().system() returns None even when the command fails, so the
    usual `if rc: raise` never fires.
    """
    print(cmd)
    rc = subprocess.run(cmd, shell=True).returncode
    if check and rc:
        raise RuntimeError(f"command failed with exit code {rc}: {cmd}")
    return rc


WS = os.environ.get("WORKSPACE_BUCKET", "").rstrip("/")
assert WS, "WORKSPACE_BUCKET is unset"

INTERIM = SCRIPTS / "felix_interim_scan.sh"
assert INTERIM.is_file(), INTERIM

# summarize_felix_results.py lives under felix/scripts/ in the bucket, which the
# scripts/ bootstrap does not cover.
SUMMARIZE = Path("summarize_felix_results.py")
if not SUMMARIZE.exists():
    sh(f"gsutil cp {WS}/felix/scripts/summarize_felix_results.py {SUMMARIZE}")
COMPARE = Path("compare_ru_arms.py")
if not COMPARE.exists():
    sh(f"gsutil cp {WS}/felix/scripts/compare_ru_arms.py {COMPARE}")
print("interim scan :", INTERIM)
print("summarizer   :", SUMMARIZE.resolve())
print("arm compare  :", COMPARE.resolve())


In [ ]:
RUN_ROOT = os.environ.get("FELIX_RUN_ROOT", "").rstrip("/")
if not RUN_ROOT:
    sid, wid = os.environ.get("FELIX_SUBMISSION_ID", ""), os.environ.get("FELIX_WORKFLOW_ID", "")
    name = os.environ.get("WORKFLOW_NAME", "FelixGenome")
    if not (sid and wid):
        raise SystemExit(
            "Set FELIX_RUN_ROOT, or FELIX_SUBMISSION_ID + FELIX_WORKFLOW_ID."
        )
    RUN_ROOT = f"{WS}/submissions/{sid}/{name}/{wid}"

PHENOTYPES = os.environ.get("FELIX_PHENOTYPES", f"{WS}/tractor_mix_pilot/selected_phenotypes.txt")
PHENO_COV = os.environ.get("FELIX_PHENO_COV", f"{WS}/tractor_mix_pilot/pheno_cov.tsv")
WORK = Path(os.environ.get("FELIX_INTERIM_DIR", "felix_interim"))
WORK.mkdir(parents=True, exist_ok=True)

MIN_MAIN = int(os.environ.get("FELIX_MIN_CHROMS_MAIN", "22"))
MIN_REPEAT = int(os.environ.get("FELIX_MIN_CHROMS_REPEAT", "1"))
ARMS = {"main": MIN_MAIN, "ru": MIN_REPEAT, "ru_biallelic": MIN_REPEAT}

print("run    :", RUN_ROOT)
print("phenos :", PHENOTYPES)
print("work   :", WORK.resolve())
print("arms   :", ARMS)


In [ ]:
import pandas as pd

cov = {}
for arm in ARMS:
    out = WORK / f"interim_{arm}"
    sh(f"bash {INTERIM} --run {RUN_ROOT} --phenotypes {PHENOTYPES} "
       f"--arm {arm} --min-chroms {ARMS[arm]} --out {out} --coverage-only", check=False)
    f = out / "coverage.tsv"
    if f.is_file():
        cov[arm] = pd.read_csv(f, sep="\t")
    else:
        print(f"  no shards yet for arm {arm}")

if not cov:
    raise SystemExit("No step-2 shards found in any arm yet.")

wide = None
for arm, df in cov.items():
    col = df.set_index("phenotype")["chromosomes_present"].rename(arm)
    wide = col.to_frame() if wide is None else wide.join(col, how="outer")
wide = wide.fillna(0).astype(int)
print("\nchromosomes present per phenotype:")
print(wide.to_string())
print("\nmax coverage per arm:", dict(wide.max()))
print("A repeat-arm number below 22 means that arm's figures are a chromosome subset.")


In [ ]:
merged = {}
for arm, minc in ARMS.items():
    if arm not in cov:
        continue
    out = WORK / f"interim_{arm}"
    # For the repeat arms take every phenotype that has any shard; coverage is
    # reported above and travels with the figures.
    sh(f"bash {INTERIM} --run {RUN_ROOT} --phenotypes {PHENOTYPES} "
       f"--arm {arm} --min-chroms {minc} --out {out}", check=False)
    files = sorted(out.glob("*.merged.felix.tsv"))
    merged[arm] = files
    print(f"{arm}: {len(files)} merged phenotype TSVs, "
          f"{sum(f.stat().st_size for f in files) / 1e9:.2f} GB")


In [ ]:
calib = {}
for arm, files in merged.items():
    if not files:
        continue
    phenos = [f.name.replace(".merged.felix.tsv", "") for f in files]
    sdir = WORK / f"interim_{arm}" / "summary"
    # summarize_felix_results.py takes no --arm; the WDL uses that only to name
    # its bundle directory.
    sh(f"python3 {SUMMARIZE} --results {' '.join(str(f) for f in files)} "
       f"--phenotypes {' '.join(phenos)} --pheno-cov {PHENO_COV} --out-dir {sdir}",
       check=False)
    f = sdir / "calibration_summary.tsv"
    if f.is_file():
        calib[arm] = pd.read_csv(f, sep="\t")

if calib:
    lam = None
    for arm, df in calib.items():
        c = [x for x in df.columns if "lambda" in x.lower()]
        if not c:
            continue
        col = df.set_index("phenotype")[c[0]].rename(arm)
        lam = col.to_frame() if lam is None else lam.join(col, how="outer")
    if lam is not None:
        print("lambda_gc by phenotype and arm "
              "(repeat arms over the chromosome subset shown earlier):")
        print(lam.round(4).to_string())
        lam.to_csv(WORK / "interim_lambda_gc.tsv", sep="\t")
    for arm, df in calib.items():
        keep = [c for c in ("phenotype", "n_variants", "n_cases", "n_controls") if c in df.columns]
        if keep:
            print(f"\n{arm}:")
            print(df[keep].to_string(index=False))


In [ ]:
from IPython.display import Image, display

for arm in ARMS:
    qc = WORK / f"interim_{arm}" / "summary" / "qc"
    if not qc.is_dir():
        continue
    pngs = sorted(qc.glob("*/qq_cct.png")) + sorted(qc.glob("*/manhattan_cct.png"))
    print(f"=== {arm}: {len(pngs)} plots ===")
    for p in pngs[:8]:
        print(f"{arm} / {p.parent.name} / {p.name}")
        display(Image(filename=str(p)))


In [ ]:
# --- Four-arm (2x2) comparison: local ancestry x repeat coding --------------
# This is the comparison the biallelic arm exists for, and the direct input to
# Figure 3's dosage-vs-biallelic panel. It uses only the two REPEAT arms. The
# main arm is not a comparator here: its markers are individual VCF records,
# not catalog loci, so they do not join to these (felix/REPEAT_DOSAGE.md 6.2).
#
#   A pooled_biallelic   carrier coding, p.value_ancALL   <- conventional GWAS
#   B la_biallelic       carrier coding, P_cct_admixed_c
#   C pooled_dosage      ru coding,      p.value_ancALL
#   D la_dosage          ru coding,      P_cct_admixed_c     <- production arm
#
# Repeat-dosage benefit is D vs B (and C vs A); local-ancestry benefit is
# D vs C (and B vs A). Every contrast is paired at the locus, so a chromosome
# subset costs precision, not validity. Significance COUNTS are not
# interpretable on a subset, and "more hits" is not a benefit claim even on a
# complete run without matched lambda (felix/REPEAT_DOSAGE.md 3, 8).
import json as _json
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
from IPython.display import Image, display

ARM_OUT = WORK / "arms"
ARM_OUT.mkdir(parents=True, exist_ok=True)

pairs = []
for f in merged.get("ru", []):
    g = WORK / "interim_ru_biallelic" / f.name
    if g.is_file():
        pairs.append((f.name.replace(".merged.felix.tsv", ""), f, g))
print(f"phenotypes with both repeat arms merged: {len(pairs)}")

summaries = []
for pheno, dos, bia in pairs:
    js = ARM_OUT / f"{pheno}.arms.json"
    rc = sh(f"python3 {COMPARE} --dosage-tsv {dos} --biallelic-tsv {bia} "
            f"--phenotype {pheno} --out-tsv {ARM_OUT / (pheno + '.arms.tsv')} "
            f"--out-summary-json {js} --out-md {ARM_OUT / (pheno + '.arms.md')}",
            check=False)
    if rc == 0 and js.is_file():
        summaries.append(_json.loads(js.read_text()))

if not summaries:
    print("no phenotype has both repeat arms merged yet; the grid needs RuStep2 "
          "and BaStep2 to have finished on the same chromosome for one phenotype")
else:
    rows = [
        {"phenotype": s["phenotype"], "arm": arm, "coding": d["coding"],
         "ancestry": d["ancestry"], "n_tested": d["n_tested"],
         "lambda": d["lambda"], "n_significant": d["n_significant"]}
        for s in summaries for arm, d in s["arms"].items()
    ]
    arms_tbl = pd.DataFrame(rows)
    arms_tbl.to_csv(WORK / "interim_four_arm.tsv", sep="\t", index=False)
    print("\nlambda by arm (repeat loci, chromosome subset shown earlier):")
    print(arms_tbl.pivot(index="phenotype", columns="arm", values="lambda")
          .round(4).to_string())
    print("\nloci tested per arm:")
    print(arms_tbl.pivot(index="phenotype", columns="arm", values="n_tested").to_string())

    setdiff = pd.DataFrame([
        {"phenotype": s["phenotype"], "shared": s["n_loci_shared"],
         "biallelic_only": s["n_loci_biallelic_only"],
         "dosage_only": s["n_loci_dosage_only"]} for s in summaries])
    print("\nlocus sets at the min-carrier boundary (grid is on the intersection):")
    print(setdiff.to_string(index=False))

    keep = ["CHR", "POS", "MarkerID", "P_pooled_biallelic", "P_la_biallelic",
            "P_pooled_dosage", "P_la_dosage"]
    big = pd.concat(
        [pd.read_csv(ARM_OUT / f"{s['phenotype']}.arms.tsv", sep="\t",
                     usecols=keep, low_memory=False).assign(phenotype=s["phenotype"])
         for s in summaries], ignore_index=True)
    big.to_csv(WORK / "interim_four_arm_loci.tsv.gz", sep="\t", index=False)

    def nl(col):
        v = pd.to_numeric(big[col], errors="coerce").to_numpy(dtype=float)
        return -np.log10(np.clip(v, 1e-300, 1.0))

    panels = [
        ("repeat dosage benefit (D vs B)", "P_la_biallelic", "P_la_dosage",
         "-log10 p, carrier coding", "-log10 p, repeat dosage"),
        ("local ancestry benefit (D vs C)", "P_pooled_dosage", "P_la_dosage",
         "-log10 p, pooled", "-log10 p, ancestry-aware"),
    ]
    fig, axes = plt.subplots(1, 2, figsize=(10, 5))
    for ax, (title, cx, cy, xl, yl) in zip(axes, panels):
        x, y = nl(cx), nl(cy)
        ok = np.isfinite(x) & np.isfinite(y)
        ax.scatter(x[ok], y[ok], s=3, alpha=0.25, edgecolors="none")
        lim = float(max(x[ok].max(), y[ok].max())) * 1.05 if ok.any() else 1.0
        ax.plot([0, lim], [0, lim], lw=0.8, color="0.4")
        ax.set_xlabel(xl)
        ax.set_ylabel(yl)
        ax.set_title(title, fontsize=10)
        print(f"{title}: n={int(ok.sum()):,}; y stronger by >1 log unit: "
              f"{int(((y - x) > 1)[ok].sum()):,}; weaker: "
              f"{int(((x - y) > 1)[ok].sum()):,}")
    fig.suptitle(f"{len(summaries)} phenotypes, repeat loci, chromosome subset",
                 fontsize=10)
    fig.tight_layout()
    fig.savefig(WORK / "interim_four_arm.png", dpi=150)
    display(Image(filename=str(WORK / "interim_four_arm.png")))

    print("\n" + summaries[0]["notes"]["la_p_not_1df"])
    print("\n" + summaries[0]["notes"]["marker_set"])


In [ ]:
lines = ["# Interim FelixGenome results", "", f"Run: `{RUN_ROOT}`", ""]
lines += ["## Chromosome coverage per phenotype and arm", "", "```", wide.to_string(), "```", ""]
if "lam" in dir() and lam is not None:
    lines += ["## lambda_gc by arm", "",
              "Repeat-arm values are over the chromosome subset above, not genome-wide.",
              "", "```", lam.round(4).to_string(), "```", ""]
if "arms_tbl" in dir():
    lines += ["## Four-arm grid (repeat loci only)", "",
              "A = carrier/pooled is the conventional-GWAS baseline; D = dosage/"
              "ancestry-aware is the production arm. Paired at the locus, so the "
              "chromosome subset costs precision, not validity. Counts of "
              "significant loci are not interpretable on a subset.", "",
              "```", arms_tbl.pivot(index="phenotype", columns="arm",
                                    values="lambda").round(4).to_string(), "```", ""]
    if "setdiff" in dir():
        lines += ["Locus sets at the min-carrier boundary:", "",
                  "```", setdiff.to_string(index=False), "```", ""]
outs = sorted(p.name for p in WORK.glob("interim_*"))
lines += ["## Files", ""] + [f"- `{n}`" for n in outs]
note = WORK / "interim_review.md"
note.write_text("\n".join(lines) + "\n")
print(note)
print("\n".join(lines))
